# Homework 5 Batch

In [53]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, max, unix_timestamp, count
import os

In [39]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework') \
    .getOrCreate()

In [40]:
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

In [41]:
df = spark.read.parquet("yellow_tripdata_2024-10.parquet")

## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?
- 3.5.5

In [42]:
spark.version

'3.5.5'

## Question 2: Yellow October 2024

Read the October 2024 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB <-
- 75MB
- 100MB

In [43]:
df_part = df.repartition(4)

df_part.write.parquet("homework", mode="overwrite")

In [44]:
output_dir = "./homework"
parquet_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.parquet')]
file_sizes_mb = [os.path.getsize(f) / (1024 * 1024) for f in parquet_files]
average_size_mb = sum(file_sizes_mb) / len(file_sizes_mb)

print(f"Average size of Parquet files: {average_size_mb:.2f} MB")

Average size of Parquet files: 22.40 MB


## Question 3: Count records 

How many taxi trips were there on the 15th of October?

Consider only trips that started on the 15th of October.

- 85,567
- 105,567
- 125,567 <-
- 145,567

In [45]:
df_trips = df.filter(to_date(col("tpep_pickup_datetime")) == "2024-10-15").count()

print(f'On 15th of October 2024 there were {df_trips} trips.')

On 15th of October 2024 there were 128893 trips.


## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 122
- 142
- 162 <-
- 182

In [46]:
df_max = df.withColumn(
    "trip_duration_hours",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
)

max_trip_duration = df_max.select(max("trip_duration_hours")).collect()[0][0]

In [48]:
print(f"The longest trip was done in {max_trip_duration:.2f} hours")

The longest trip was done in 162.62 hours


## Question 5: User Interface

Spark’s User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040 <-
- 8080

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

- Governor's Island/Ellis Island/Liberty Island <-
- Arden Heights
- Rikers Island
- Jamaica Bay

In [ ]:
# !wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-05 15:39:30--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2123:4000:b:20a5:b140:21, 2600:9000:2123:3c00:b:20a5:b140:21, 2600:9000:2123:9200:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2123:4000:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-05 15:39:31 (1.61 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [ ]:
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True)

In [56]:
df_pickup_counts = df.groupBy("PULocationID").agg(count("*").alias("pickup_count"))

In [57]:
df_pickup_zones = df_pickup_counts.join(
    df_zones,
    df_pickup_counts.PULocationID == df_zones.LocationID,
    "inner"
).select("Zone", "pickup_count")

least_frequent = df_pickup_zones.orderBy("pickup_count").first()

In [60]:
print(f"The least frequent pickup zone is: {least_frequent['Zone']}")

The least frequent pickup zone is: Governor's Island/Ellis Island/Liberty Island


In [61]:
spark.stop()